# Part 7: Callbacks, Context & Memory
## Content Creation Studio Workshop

Welcome back! In Parts 1-6, you built agents, gave them tools, orchestrated teams, and mastered sequential, iterative, and parallel workflows. Now it's time to add the **operational layer** that makes agent systems robust and stateful.

In this part, we'll learn how to:
- **Monitor and control** agent execution with callbacks
- **Share data** between agents using sessions and state
- **Persist outputs** as artifacts that survive beyond sessions
- **Remember across sessions** with long-term memory

These are the building blocks you'll need for the capstone project in Part 8.

---

## The Content Creation Studio Playbook Series
- Part 1: Building Your First AI Agent 
- Part 2: Giving Your Agent Custom Tools 
- Part 3: Building Agent Teams with Agent-as-a-Tool 
- Part 4: Sequential Workflows & Design Patterns 
- Part 5: Building Iterative Workflows with LoopAgent 
- Part 6: Efficient Workflows with ParallelAgent 
- **Part 7: Callbacks, Context & Memory** (You are here) 
- Part 8: The Capstone Project
- Part 9: Deploying to Vertex AI Agent Engine

---

## New Concepts in This Part

In Part 7, we'll introduce these **new ADK concepts**:

1. **Callbacks** - Hook into agent and model lifecycle events for logging, guardrails, and metrics
2. **Sessions & State** - Share data between agents within a conversation using `output_key` and `{{variable}}`
3. **Artifacts** - Save and load persistent files (text or binary) during a session
4. **Memory** - Enable agents to recall information from past sessions

Each concept will be clearly marked with when first introduced!

## 1. Setup: Install Libraries

In [ ]:
!pip install google-adk==1.24.1 -q


## 2. Authentication

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key
print(" API Key configured successfully!")


## Section 1: Callbacks

### NEW CONCEPT: Callbacks

> **What are Callbacks?**
> Callbacks are functions you attach to an agent that fire at specific points in the agent's lifecycle. They let you add logging, guardrails, metrics, and custom logic **without modifying agent instructions**.
>
> ADK provides **four callback types**:
>
> | Callback | When It Fires | Common Use Cases |
> |---|---|---|
> | `before_agent_callback` | Before the agent starts processing | Logging, timing, setup |
> | `after_agent_callback` | After the agent finishes processing | Logging completion time, cleanup |
> | `before_model_callback` | Before the request is sent to the LLM | Content safety guardrails, request modification |
> | `after_model_callback` | After the LLM returns a response | Output logging, metrics, response modification |
>
> **Signatures**:
> ```python
> def before_agent_callback(callback_context: CallbackContext) -> Optional[Content]:
>     # Return None to proceed normally, or Content to skip the agent
>
> def after_agent_callback(callback_context: CallbackContext) -> Optional[Content]:
>     # Return None to proceed normally, or Content to override the response
>
> def before_model_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
>     # Return None to proceed normally, or LlmResponse to skip the model call
>
> def after_model_callback(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
>     # Return None to proceed normally, or LlmResponse to override the response
> ```
>
> **Key Insight**: Returning `None` means "proceed normally." Returning a value means "use this instead" (short-circuit).
>
> **Reference**: [Callbacks](https://google.github.io/adk-docs/agents/callbacks/)

Let's define four callback functions that add timing, guardrails, and output metrics to our agents.

### Callback Flow Visualization

The official ADK documentation provides this visualization of where callbacks
fire in the agent execution lifecycle:

![ADK Callback Flow](https://google.github.io/adk-docs/assets/callback_flow.png)

*Source: [Google ADK Callbacks Documentation](https://google.github.io/adk-docs/callbacks/)*

As shown in the diagram, callbacks wrap the agent, model, and tool execution
at well-defined points. Returning `None` proceeds normally; returning a value
short-circuits (skips) the wrapped step.

In [ ]:
import time
from typing import Optional
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmResponse, LlmRequest
from google.genai import types

BLOCKED_TOPICS = ["violence", "illegal", "harmful", "hate speech"]
_agent_start_times = {}


# Callback 1: Fires BEFORE the agent starts
def before_agent_callback(callback_context: CallbackContext) -> Optional[types.Content]:
    """Logs agent start and tracks execution time."""
    agent_name = callback_context.agent_name
    _agent_start_times[agent_name] = time.time()
    print(f"\n{'---'*20}")
    print(f" AGENT START: {agent_name}")
    print(f"{'---'*20}")
    return None  # Proceed normally


# Callback 2: Fires AFTER the agent finishes
def after_agent_callback(callback_context: CallbackContext) -> Optional[types.Content]:
    """Logs agent completion with execution time."""
    agent_name = callback_context.agent_name
    elapsed = time.time() - _agent_start_times.pop(agent_name, time.time())
    print(f"\n{'---'*20}")
    print(f" AGENT DONE: {agent_name} ({elapsed:.1f}s)")
    print(f"{'---'*20}")
    return None


# Callback 3: Fires BEFORE the model (LLM) is called
def before_model_callback(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Content safety guardrail -- blocks prohibited topics before they reach the model."""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.parts:
            text = (last.parts[0].text or "").lower()
            for topic in BLOCKED_TOPICS:
                if topic in text:
                    print(f" GUARDRAIL: Blocked '{topic}'")
                    return LlmResponse(
                        content=types.Content(
                            parts=[
                                types.Part.from_text(
                                    text=f"I cannot generate content about '{topic}'."
                                )
                            ],
                            role="model",
                        )
                    )
    return None  # Proceed normally


# Callback 4: Fires AFTER the model returns a response
def after_model_callback(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """Logs model output word count."""
    if llm_response.content and llm_response.content.parts:
        text = llm_response.content.parts[0].text or ""
        print(
            f" Model output for {callback_context.agent_name}: ~{len(text.split())} words"
        )
    return None


print(" All 4 callback functions defined!")


Now let's create a content drafter agent with all four callbacks wired up, and test it.

In [ ]:
from google.adk.agents import Agent

content_drafter_agent = Agent(
    name="content_drafter_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a blog content drafter. Write a short blog intro (100-150 words)
    on the topic the user provides. Output only the intro in markdown.
    """,
    tools=[],
    before_agent_callback=before_agent_callback,
    after_agent_callback=after_agent_callback,
    before_model_callback=before_model_callback,
    after_model_callback=after_model_callback,
)

print(f" Agent '{content_drafter_agent.name}' created with all 4 callbacks!")


In [ ]:
from IPython.display import display, Markdown
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai.types import Content, Part

session_service = InMemorySessionService()
user_id = "adk_content_creator_001"


async def run_agent_query(agent, query, session, user_id):
    """Run a query against an agent and display the result."""
    runner = Runner(agent=agent, session_service=session_service, app_name=agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
        ):
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"Error: {e}"
    print("\n" + "-" * 50)
    display(Markdown(final_response))
    print("-" * 50)
    return final_response


# --- Test 1: Normal query (observe timing logs) ---
print("=" * 60)
print(" Test 1: Normal query (timing + output metrics)")
print("=" * 60)
session1 = await session_service.create_session(
    app_name=content_drafter_agent.name, user_id=user_id
)
await run_agent_query(
    content_drafter_agent,
    "Write a blog intro about sustainable fashion trends",
    session1,
    user_id,
)

# --- Test 2: Blocked topic (guardrail triggers) ---
print("\n" + "=" * 60)
print(" Test 2: Blocked topic (guardrail in before_model_callback)")
print("=" * 60)
session2 = await session_service.create_session(
    app_name=content_drafter_agent.name, user_id=user_id
)
await run_agent_query(
    content_drafter_agent, "Write content about violence in media", session2, user_id
)


**What happened:**

- **Test 1**: All four callbacks fired in order: `before_agent` (start timer) -> `before_model` (no block) -> `after_model` (log word count) -> `after_agent` (log elapsed time). The agent produced content normally.
- **Test 2**: The `before_model_callback` detected a blocked topic and returned an `LlmResponse` directly, **skipping the LLM call entirely**. This is how guardrails work -- they short-circuit the pipeline.

---

## Section 2: Sessions & State

### NEW CONCEPT: Sessions & State for Agent Data Sharing

> **How Do Agents Share Data Within a Conversation?**
>
> ADK provides two key mechanisms:
>
> 1. **Sessions** -- The conversation container. A session holds the message history and a key-value **state** dictionary.
> 2. **State** -- Key-value data shared between agents within a session. Agents write to state via `output_key` and read from it via `{{variable}}` interpolation.
>
> **Writing state** -- use `output_key`:
> ```python
> agent = Agent(
>     instruction="Research topics about: {{content_brief}}",
>     output_key="blog_topic"  # Response saved to state["blog_topic"]
> )
> ```
>
> **Reading state** -- use `{{variable}}` in instructions:
> ```python
> agent = Agent(
>     instruction="Write about: {{blog_topic}}"  # Reads state["blog_topic"]
> )
> ```
>
> **How state flows in a SequentialAgent:**
> ```
> Agent A (output_key="blog_topic")
>  -> state["blog_topic"] = "10 AI Tools for Remote Workers"
>  |
> Agent B (instruction uses {{blog_topic}})
>  -> Reads: "Write about: 10 AI Tools for Remote Workers"
>  -> output_key="blog_content"
>  |
> Agent C (instruction uses {{blog_content}})
>  -> Reads the draft content
> ```
>
> **Key Points**:
> - Each session has a unique `session.id`
> - You can initialize state at creation time: `state={"key": "value"}`
> - `InMemorySessionService` stores sessions in memory (development only)
> - State is lost when the session ends
>
> **Reference**: [Sessions & State](https://google.github.io/adk-docs/sessions/)

Let's demonstrate state passing with a 2-agent sequential workflow: a topic researcher that stores its result, and a content writer that reads it.

In [ ]:
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search

# Agent 1: Researches a topic and stores the result via output_key
topic_researcher = Agent(
    name="topic_researcher",
    model="gemini-2.5-flash",
    instruction="""
    You are a content strategist. Find one compelling blog topic about
    sustainable living. Output ONLY the title, nothing else.
    Example: "10 Zero-Waste Swaps to Transform Your Kitchen"
    """,
    tools=[google_search],
    output_key="blog_topic",  # Stores result in state["blog_topic"]
)

# Agent 2: Reads {{blog_topic}} from state and writes a short draft
content_writer = Agent(
    name="content_writer",
    model="gemini-2.5-flash",
    instruction="""
    Write a short blog intro (100-150 words) about: {{blog_topic}}
    Output only the intro in markdown.
    """,
    tools=[],
    output_key="blog_content",  # Stores result in state["blog_content"]
)

# Chain them in a SequentialAgent
state_demo_workflow = SequentialAgent(
    name="state_demo_workflow", sub_agents=[topic_researcher, content_writer]
)

print(" State demo workflow created!")
print(" topic_researcher (output_key='blog_topic')")
print("   -> content_writer (reads {{blog_topic}}, output_key='blog_content')")


In [ ]:
session_service_state = InMemorySessionService()


async def run_state_demo():
    session = await session_service_state.create_session(
        app_name=state_demo_workflow.name, user_id=user_id
    )

    runner = Runner(
        agent=state_demo_workflow,
        session_service=session_service_state,
        app_name=state_demo_workflow.name,
    )

    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(
            parts=[Part(text="Find a topic about sustainable living")], role="user"
        ),
    ):
        if event.is_final_response():
            final_response = event.content.parts[0].text

    # Retrieve the updated session to inspect state
    updated_session = await session_service_state.get_session(
        app_name=state_demo_workflow.name, user_id=user_id, session_id=session.id
    )

    print("\n" + "=" * 60)
    print(" Session State After Workflow:")
    print("=" * 60)
    for key, value in updated_session.state.items():
        preview = str(value)[:120] + "..." if len(str(value)) > 120 else str(value)
        print(f"  {key}: {preview}")
    print("=" * 60)

    print("\n" + "-" * 50)
    print(" Final Output (content_writer):")
    display(Markdown(final_response))
    print("-" * 50)


await run_state_demo()


**What happened:**

1. `topic_researcher` ran, found a topic, and stored it in `state["blog_topic"]` via `output_key`
2. `content_writer` read `{{blog_topic}}` from state and wrote a blog intro
3. The final `state["blog_content"]` contains the written intro

No manual data passing was needed -- ADK handled it automatically through `output_key` and `{{variable}}` interpolation.

---

## Section 3: Artifacts

### NEW CONCEPT: Artifacts

> **What are Artifacts?**
> Artifacts are files (text or binary) saved during a session. Unlike state (which is lost when the session ends), artifacts can persist and be retrieved later.
>
> **Use artifacts when you need to:**
> - Save generated content (blog posts, reports)
> - Create downloadable files
> - Persist outputs beyond the session
>
> **API**:
> ```python
> # Save an artifact (inside a tool function)
> artifact = types.Part.from_text(text=content)
> version = await tool_context.save_artifact(filename="blog_post.md", artifact=artifact)
>
> # List all artifacts
> filenames = await tool_context.list_artifacts()
>
> # Load a specific artifact
> artifact = await tool_context.load_artifact(filename="blog_post.md")
> text = artifact.text
> ```
>
> **Runner setup** -- you must provide an `artifact_service`:
> ```python
> from google.adk.artifacts import InMemoryArtifactService
>
> runner = Runner(
>     agent=my_agent,
>     session_service=session_service,
>     artifact_service=InMemoryArtifactService(),  # Enable artifacts
> )
> ```
>
> **State vs Artifacts**:
> | | **State** | **Artifacts** |
> |---|---|---|
> | **Lifetime** | Within one session | Persistent |
> | **Content** | Key-value pairs | Files (text, binary) |
> | **Access** | `{{key}}` in instructions | `tool_context.save/load_artifact()` |
> | **Use Case** | Agent-to-agent data passing | Saving generated content |
>
> **Reference**: [Artifacts](https://google.github.io/adk-docs/sessions/artifacts/)

Let's build artifact tools and an agent that can save, list, and load content as artifacts.

In [ ]:
from google.adk.tools import ToolContext
from google.genai import types


async def save_artifact(tool_context: ToolContext, filename: str, content: str) -> dict:
    """Saves text content as a named artifact. Use this to persist generated content."""
    artifact = types.Part.from_text(text=content)
    version = await tool_context.save_artifact(filename=filename, artifact=artifact)
    print(f" Artifact saved: {filename} (version {version})")
    return {"saved": filename, "version": version}


async def load_artifact(tool_context: ToolContext, filename: str) -> dict:
    """Loads a previously saved artifact by filename."""
    artifact = await tool_context.load_artifact(filename=filename)
    if artifact is None:
        return {"error": f"Artifact '{filename}' not found"}
    print(f" Artifact loaded: {filename}")
    return {"filename": filename, "content": artifact.text}


async def list_artifacts(tool_context: ToolContext) -> dict:
    """Lists all saved artifact filenames in the current session."""
    filenames = await tool_context.list_artifacts()
    print(f" Artifacts in session: {filenames}")
    return {"artifacts": filenames}


print(" Artifact tools defined: save_artifact, load_artifact, list_artifacts")


In [ ]:
from google.adk.artifacts import InMemoryArtifactService

artifact_agent = Agent(
    name="artifact_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a content manager. You can save, list, and load content artifacts.

    When asked to create and save content:
    1. Write the content
    2. Save it using `save_artifact` with an appropriate filename
    3. Confirm it was saved

    When asked to list artifacts, use `list_artifacts`.
    When asked to load an artifact, use `load_artifact` with the filename.
    """,
    tools=[save_artifact, load_artifact, list_artifacts],
)

# Create services with artifact support
session_service_art = InMemorySessionService()
artifact_service = InMemoryArtifactService()


async def run_artifact_demo():
    session = await session_service_art.create_session(
        app_name=artifact_agent.name, user_id=user_id
    )

    # Runner with artifact_service enabled
    runner = Runner(
        agent=artifact_agent,
        session_service=session_service_art,
        artifact_service=artifact_service,
        app_name=artifact_agent.name,
    )

    # Step 1: Create and save a blog post
    print("=" * 60)
    print(" Step 1: Create and save content as artifact")
    print("=" * 60)
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(
            parts=[
                Part(
                    text="Write a short blog intro (50 words) about eco-friendly packaging "
                    "and save it as 'eco_packaging_intro.md'"
                )
            ],
            role="user",
        ),
    ):
        if event.is_final_response():
            display(Markdown(event.content.parts[0].text))

    # Step 2: List saved artifacts
    print("\n" + "=" * 60)
    print(" Step 2: List all artifacts")
    print("=" * 60)
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(parts=[Part(text="List all saved artifacts")], role="user"),
    ):
        if event.is_final_response():
            display(Markdown(event.content.parts[0].text))

    # Step 3: Load the artifact back
    print("\n" + "=" * 60)
    print(" Step 3: Load artifact back")
    print("=" * 60)
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(
            parts=[
                Part(
                    text="Load the eco_packaging_intro.md artifact and show me the content"
                )
            ],
            role="user",
        ),
    ):
        if event.is_final_response():
            display(Markdown(event.content.parts[0].text))


await run_artifact_demo()


**What happened:**

1. The agent wrote content and saved it as an artifact using `save_artifact`
2. `list_artifacts` showed all saved files in the session
3. `load_artifact` retrieved the saved content by filename

Artifacts persist beyond the conversation turn. They are ideal for saving generated content that you want to retrieve later or pass to external systems.

---

## Section 4: Memory

### NEW CONCEPT: Memory (Short-Term vs Long-Term)

> **What is Memory in ADK?**
> Memory enables your agents to recall information from **past sessions**. Without memory, every new session starts from scratch.
>
> ADK distinguishes between two types:
>
> **Short-Term Memory** = Session State
> - Lives within a single session only
> - Agents share data via `output_key` / `{{variable}}`
> - Lost when the session ends
>
> **Long-Term Memory** = MemoryService
> - Persists **across sessions**
> - Stores past conversation summaries
> - Enables continuity: "Last time you asked about AI productivity..."
>
> **How Long-Term Memory Works:**
> ```
> Session 1: User creates content about AI
>  | (session saved to memory after completion)
>
> Session 2: User starts a new conversation
>  | PreloadMemoryTool auto-loads relevant past context
>  Agent knows: "You previously worked on AI productivity content"
> ```
>
> **Setup requires 3 things:**
> 1. `InMemoryMemoryService` -- the memory store
> 2. `PreloadMemoryTool` on the agent -- auto-loads relevant memories before each turn
> 3. `memory_service` passed to the Runner
>
> **Key Point**: `PreloadMemoryTool` runs **automatically** before each turn. The LLM doesn't decide to call it -- it just sees the relevant context.
>
> **Reference**: [Memory](https://google.github.io/adk-docs/sessions/memory/)

Let's demonstrate long-term memory with two sessions: the agent will remember what happened in Session 1 when we start Session 2.

In [ ]:
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import preload_memory_tool

# Agent with PreloadMemoryTool for cross-session recall
memory_agent = Agent(
    name="memory_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a content creation assistant. Help users with blog topics and content.

    Past conversations are automatically loaded before each turn.
    If you have relevant context from previous sessions, reference it naturally.
    For example: "Last time we discussed X, would you like to continue with that?"
    """,
    tools=[preload_memory_tool.PreloadMemoryTool()],
)

print(" Memory agent created with PreloadMemoryTool!")


In [ ]:
session_service_mem = InMemorySessionService()
memory_service = InMemoryMemoryService()


async def run_memory_demo():
    runner = Runner(
        agent=memory_agent,
        session_service=session_service_mem,
        memory_service=memory_service,
        app_name=memory_agent.name,
    )

    # --- Session 1: Discuss a topic ---
    print("=" * 60)
    print(" SESSION 1: Initial conversation")
    print("=" * 60)
    session1 = await session_service_mem.create_session(
        app_name=memory_agent.name, user_id=user_id
    )

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session1.id,
        new_message=Content(
            parts=[
                Part(
                    text="I'm working on a blog series about sustainable fashion. "
                    "Can you suggest a topic for my first post?"
                )
            ],
            role="user",
        ),
    ):
        if event.is_final_response():
            display(Markdown(event.content.parts[0].text))

    # Save Session 1 to long-term memory
    updated_session1 = await session_service_mem.get_session(
        app_name=memory_agent.name, user_id=user_id, session_id=session1.id
    )
    await memory_service.add_session_to_memory(updated_session1)
    print("\n Session 1 saved to long-term memory.")

    # --- Session 2: New session, agent should recall Session 1 ---
    print("\n" + "=" * 60)
    print(" SESSION 2: New session (agent should recall past context)")
    print("=" * 60)
    session2 = await session_service_mem.create_session(
        app_name=memory_agent.name, user_id=user_id
    )

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session2.id,
        new_message=Content(
            parts=[Part(text="What should I write about next for my blog?")],
            role="user",
        ),
    ):
        if event.is_final_response():
            display(Markdown(event.content.parts[0].text))


await run_memory_demo()


**What happened:**

1. **Session 1**: We discussed sustainable fashion topics. After the session ended, we saved it to `memory_service`.
2. **Session 2**: The agent received a vague question ("What should I write about next?"). Thanks to `PreloadMemoryTool`, it automatically loaded relevant context from Session 1 and referenced the sustainable fashion blog series.

Without memory, Session 2 would have no idea what "my blog" refers to.

---

## Comparison: State vs Artifacts vs Memory

### Visual: Data Persistence Layers

```
┌─────────────────────────────────────────────────────────┐
│                    Session Scope                         │
│                                                          │
│  ┌─────────────────────────────────────────────────┐    │
│  │              STATE (key-value)                   │    │
│  │                                                  │    │
│  │  Agent A ──output_key──> state["blog_topic"]     │    │
│  │                              │                   │    │
│  │  Agent B ──{{blog_topic}}────┘                   │    │
│  │                                                  │    │
│  │  Lifetime: within one session only               │    │
│  └─────────────────────────────────────────────────┘    │
│                                                          │
│  ┌─────────────────────────────────────────────────┐    │
│  │             ARTIFACTS (files)                    │    │
│  │                                                  │    │
│  │  save_artifact("blog.md", content)               │    │
│  │  load_artifact("blog.md") --> content            │    │
│  │                                                  │    │
│  │  Lifetime: persists beyond session               │    │
│  └─────────────────────────────────────────────────┘    │
│                                                          │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│                Cross-Session Scope                       │
│                                                          │
│  ┌─────────────────────────────────────────────────┐    │
│  │              MEMORY (recall)                     │    │
│  │                                                  │    │
│  │  Session 1 ──save──> MemoryService               │    │
│  │                          │                       │    │
│  │  Session 2 <──PreloadMemoryTool──┘               │    │
│  │  "Last time you asked about sustainable fashion" │    │
│  │                                                  │    │
│  │  Lifetime: persists across all sessions          │    │
│  └─────────────────────────────────────────────────┘    │
│                                                          │
└─────────────────────────────────────────────────────────┘
```

| | **State** | **Artifacts** | **Memory** |
|---|---|---|---|
| **Scope** | Within one session | Persistent across sessions | Persistent across sessions |
| **Content** | Key-value pairs | Files (text, binary) | Conversation summaries |
| **Access** | `{{key}}` in instructions | `tool_context.save/load_artifact()` | Auto-loaded by `PreloadMemoryTool` |
| **Persistence** | Lost when session ends | Survives session end | Survives session end |
| **Use Case** | Agent-to-agent data passing | Saving generated files | Conversational continuity |
| **Example** | `blog_topic`, `current_content` | `blog_post.md`, `report.pdf` | "Last time you asked about AI..." |

## Combined Example: All Features Together

Let's bring callbacks, state, artifacts, and memory together in one workflow. This mirrors the architecture you'll build in the Part 8 capstone:

- **Callbacks** on the agent for timing and guardrails
- **State** to pass data between sequential agents
- **Artifacts** to save the final output
- **Memory** to enable cross-session recall

In [ ]:
# Combined demo: callbacks + state + artifacts + memory

# Agent 1: Research a topic (with callbacks for timing)
combined_researcher = Agent(
    name="combined_researcher",
    model="gemini-2.5-flash",
    instruction="""
    Find one compelling blog topic about eco-friendly technology.
    Output ONLY the title, nothing else.
    """,
    tools=[google_search],
    output_key="blog_topic",
    before_agent_callback=before_agent_callback,
    after_agent_callback=after_agent_callback,
    before_model_callback=before_model_callback,
    after_model_callback=after_model_callback,
)

# Agent 2: Write content using state, save as artifact (with callbacks)
combined_writer = Agent(
    name="combined_writer",
    model="gemini-2.5-flash",
    instruction="""
    Write a blog intro (100-150 words) about: {{blog_topic}}

    After writing, save the content as an artifact with filename 'blog_intro.md'
    using the save_artifact tool.

    Output the content you wrote.
    """,
    tools=[save_artifact],
    output_key="blog_content",
    before_agent_callback=before_agent_callback,
    after_agent_callback=after_agent_callback,
    after_model_callback=after_model_callback,
)

combined_workflow = SequentialAgent(
    name="combined_workflow",
    sub_agents=[combined_researcher, combined_writer],
)

# Services with artifact + memory support
session_service_combined = InMemorySessionService()
artifact_service_combined = InMemoryArtifactService()
memory_service_combined = InMemoryMemoryService()


async def run_combined_demo():
    session = await session_service_combined.create_session(
        app_name=combined_workflow.name, user_id=user_id
    )

    runner = Runner(
        agent=combined_workflow,
        session_service=session_service_combined,
        artifact_service=artifact_service_combined,
        memory_service=memory_service_combined,
        app_name=combined_workflow.name,
    )

    print("=" * 60)
    print(" Combined Demo: Callbacks + State + Artifacts + Memory")
    print("=" * 60)

    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(
            parts=[Part(text="Create content about eco-friendly technology")],
            role="user",
        ),
    ):
        if event.is_final_response():
            final_response = event.content.parts[0].text

    # Show results
    updated_session = await session_service_combined.get_session(
        app_name=combined_workflow.name, user_id=user_id, session_id=session.id
    )

    print("\n" + "=" * 60)
    print(" Results:")
    print("=" * 60)
    print(
        f"  State - blog_topic: {updated_session.state.get('blog_topic', 'N/A')[:80]}..."
    )
    print(
        f"  State - blog_content: {str(updated_session.state.get('blog_content', 'N/A'))[:80]}..."
    )
    print("\n Final Output:")
    display(Markdown(final_response))

    # Save to memory for future sessions
    await memory_service_combined.add_session_to_memory(updated_session)
    print("\n Session saved to long-term memory for future recall.")


await run_combined_demo()


## Recap: What We've Learned

This part introduced four essential concepts for building robust agent systems:

### Core Concepts Introduced:

1. **Callbacks** - Hook into agent and model lifecycle events 
   4 types: `before_agent`, `after_agent`, `before_model`, `after_model` 
   Use for: timing, guardrails, metrics, logging 
   [Callbacks](https://google.github.io/adk-docs/agents/callbacks/)

2. **Sessions & State** - Share data between agents within a conversation 
   Write: `output_key="variable"` | Read: `{{variable}}` 
   [Sessions](https://google.github.io/adk-docs/sessions/)

3. **Artifacts** - Save and load persistent files during a session 
   API: `save_artifact()`, `load_artifact()`, `list_artifacts()` 
   Requires: `InMemoryArtifactService` in Runner 
   [Artifacts](https://google.github.io/adk-docs/sessions/artifacts/)

4. **Memory** - Enable agents to recall past sessions 
   Setup: `InMemoryMemoryService` + `PreloadMemoryTool` 
   Auto-loads relevant context before each turn 
   [Memory](https://google.github.io/adk-docs/sessions/memory/)

### Key Takeaways:

- **Callbacks** provide operational control without changing agent logic
- **State** is for within-session agent-to-agent data flow
- **Artifacts** are for persisting generated content as files
- **Memory** is for cross-session conversational continuity
- All four features compose naturally and are used together in real systems

---

## What's Next?

You now have all the building blocks: agents, tools, orchestration, sequential/loop/parallel workflows, callbacks, state, artifacts, and memory.

In **Part 8: The Capstone Project**, we'll combine **everything** from Parts 1-7 into a complete, autonomous Content Creation Studio:
- Hierarchical orchestration with a master agent
- Sequential research-and-draft pipeline
- Iterative quality improvement loop
- Parallel multi-channel content creation
- Callbacks for guardrails and timing
- Memory for cross-session context

### Preview of Part 8 Concepts:
- Hierarchical Orchestration
- Composable Agent Systems
- End-to-End Autonomous Workflows

See you in Part 8!